# 02 — DANN Prototype Training

This notebook implements the core domain-adversarial training experiment.

**Hypothesis:** A DANN trained on source (Spanish CGM) data with target (Ghanaian proxy)
data should learn domain-invariant features enabling zero-shot transfer.



In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from adamed.data.synthetic_generator import ClinicalTimeSeriesGenerator
from adamed.data.preprocessing import simple_dataloader
from adamed.models.dann import create_dann_for_adamed
from adamed.models.utils import count_parameters, get_parameter_summary
from adamed.training.trainer import DANNTrainer
from adamed.evaluation.visualization import plot_training_curves, plot_feature_space

sns.set_style('whitegrid')
print('Setup complete.')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Data Preparation

Generate synthetic data and create PyTorch DataLoader.
The data is flattened from (batch, 48, 5) to (batch, 240) for the MLP architecture.

In [ ]:
gen = ClinicalTimeSeriesGenerator(n_source=1000, n_target_proxy=200, seed=42)
data = gen.generate_experimental_split()

print(f'Total samples: {data["X"].shape[0]}')
print(f'Source: {len(data["source_idx"])} patients')
print(f'Target: {len(data["target_idx"])} patients')
print(f'Input shape per patient: {data["X"].shape[1:]} (time_steps x features)')
print(f'Flattened input dim: {data["X"].shape[1] * data["X"].shape[2]}')

loader = simple_dataloader(data, batch_size=64, normalize=True, flatten=True)
print(f'DataLoader batches: {len(loader)}')

## 2. Model Architecture

The DANN has three components:
- **Feature Extractor**: 240 -> 256 -> 128 (shared representation)
- **Label Predictor**: 128 -> 64 -> 2 (task head)
- **Domain Classifier**: 128 -> GRL -> 64 -> 2 (adversarial head)

In [ ]:
model = create_dann_for_adamed(time_steps=48, n_features=5)

print('Model Architecture:')
print(model)
print(f'\nTotal parameters: {count_parameters(model):,}')
print('\nPer-module breakdown:')
for name, count in get_parameter_summary(model).items():
    print(f'  {name}: {count:,}')

## 3. Training

Train with Ganin alpha schedule (0 -> 1 over epochs), Adam optimizer lr=1e-3.

In [ ]:
trainer = DANNTrainer(model, lr=1e-3, weight_decay=1e-4, lambda_domain=1.0)

history = trainer.train(
    loader, n_epochs=50,
    save_dir='../logs/experiment_001',
    verbose=True,
)

print(f'\nFinal label accuracy: {history["label_acc"][-1]:.4f}')
print(f'Final domain accuracy: {history["domain_acc"][-1]:.4f}')
print(f'Final alpha: {history["alpha"][-1]:.4f}')

## 4. Training Curve Analysis

In [ ]:
fig = plot_training_curves(history, save_path='../logs/experiment_001/plots/loss_curves.png')
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(history['domain_loss'], history['label_loss'],
           c=range(len(history['domain_loss'])), cmap='viridis', s=30, alpha=0.7)
ax.set_xlabel('Domain Loss', fontsize=12)
ax.set_ylabel('Label Loss', fontsize=12)
ax.set_title('Label Loss vs Domain Loss (color = epoch)', fontsize=14)
plt.colorbar(ax.collections[0], ax=ax, label='Epoch')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Feature Space Visualization

In [ ]:
eval_results = trainer.evaluate(loader)

print(f'Evaluation label accuracy: {eval_results["label_acc"]:.4f}')
print(f'Evaluation domain accuracy: {eval_results["domain_acc"]:.4f}')

fig = plot_feature_space(
    eval_results['features'], eval_results['domains'],
    save_path='../logs/experiment_001/plots/feature_tsne.png',
    title='Feature Space After DANN Training (t-SNE)',
)
plt.show()

## 6. Observations

### Key Findings:
1. **Label accuracy degrades** from ~60% to near-random (50%) as alpha increases
2. **Domain classifier** achieves high accuracy easily (domains are obviously different)
3. **Feature space** shows no domain alignment
4. **Gradient reversal** without target samples pushes features away from source distribution

### Conclusion:
Standard DANN is **not suitable for zero-shot domain adaptation**.  
See notebook 03 for detailed failure analysis.